In [ ]:
from dspy.datasets.gsm8k import GSM8K, gsm8k_metric

# Load math questions from the GSM8K dataset
gms8k = GSM8K()
trainset, devset = gms8k.train, gms8k.devA

c:\Users\jayit\GenerativeAI\dspy\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 1319/1319 [00:00<00:00, 18232.38it/s]


In [1]:
import mlflow

# Enabling tracing for DSPy
mlflow.dspy.autolog(log_compiles=True, log_evals=True, log_traces_from_compile=True)

# Optional: Set a tracking URI and an experiment
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("DSPy-optimizers")

2025/10/14 20:22:14 INFO mlflow.tracking.fluent: Experiment with name 'DSPy-optimizers' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/949896493763218950', creation_time=1760453534076, experiment_id='949896493763218950', last_update_time=1760453534076, lifecycle_stage='active', name='DSPy-optimizers', tags={}>

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()
import dspy

# groq_api_key = os.getenv("GROQ_API_KEY")

# dspy.configure(
#     lm=dspy.LM(
#         model='qwen/qwen3-32b', # Use a Groq-hosted model
#         api_key=groq_api_key,    # Replace with your actual Groq API key
#         api_base='https://api.groq.com/openai/v1' # Set the Groq API base URL
#     )
# )

llm = dspy.LM('ollama_chat/llama3.2', api_base='http://localhost:11434', api_key='', temperature = 0.2)
dspy.configure(lm=llm)
dspy.configure_cache(enable_memory_cache=False, enable_disk_cache=False)


In [4]:
simple_model = dspy.Predict("question -> answer: int", cache = False)
simple_model(question="I have 5 different balls and I randomly select 4. How many possible combinations of the balls I can get?")

2025/10/14 20:22:57 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/14 20:23:14 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/14 20:23:14 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/14 20:23:14 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/14 20:23:14 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/14 20:23:16 WARNING mlflow.traci

Prediction(
    answer=5
)

Trace(trace_id=tr-f694cf798deb83de241a099de9f8a855)

In [5]:
class CoT(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.ChainOfThought("question -> answer")
    
    def forward(self, question):
        return self.prog(question=question)

In [ ]:
from dspy.teleprompt import BootstrapFewShotWithRandomSearch

# Set up the optimizer: we want to "bootstrap" (i.e., self-generate) 8-shot examples of our CoT program.
# The optimizer will repeat this 10 times (plus some initial attempts) before selecting its best attempt on the devset.
config = dict(max_bootstrapped_demos=8, max_labeled_demos=8, num_candidate_programs=10, num_threads=4)

# Optimize! Use the `gms8k_metric` here. In general, the metric is going to tell the optimizer how well it's doing.
teleprompter = BootstrapFewShotWithRandomSearch(metric=gsm8k_metric, **config)
optimized_cot = teleprompter.compile(CoT(), trainset=trainset, valset=devset)

In [ ]:
turbo.inspect_history(n=1)




Given the fields `question`, produce the fields `answer`.

---

Follow the following format.

Question: ${question}
Reasoning: Let's think step by step in order to ${produce the answer}. We ...
Answer: ${answer}

---

Question: Andrew works in a company that provides a generous vacation allotment: for every 10 days worked, you get 1 vacation day. If last year Andrew worked 300 days and took 5 days off in March and twice as many in September, how many more vacation days can Andrew still take?
Answer: 15

Question: Frank is making hamburgers and he wants to sell them to make $50. Frank is selling each hamburger for $5 and 2 people purchased 4 and another 2 customers purchased 2 hamburgers. How many more hamburgers does Frank need to sell to make $50?
Answer: 4

Question: There are 40 students in a class. If 1/10 are absent, 3/4 of the students who are present are in the classroom, and the rest are in the canteen, how many students are in the canteen?
Answer: 9

Question: Betty and Dor

"\n\n\nGiven the fields `question`, produce the fields `answer`.\n\n---\n\nFollow the following format.\n\nQuestion: ${question}\nReasoning: Let's think step by step in order to ${produce the answer}. We ...\nAnswer: ${answer}\n\n---\n\nQuestion: Andrew works in a company that provides a generous vacation allotment: for every 10 days worked, you get 1 vacation day. If last year Andrew worked 300 days and took 5 days off in March and twice as many in September, how many more vacation days can Andrew still take?\nAnswer: 15\n\nQuestion: Frank is making hamburgers and he wants to sell them to make $50. Frank is selling each hamburger for $5 and 2 people purchased 4 and another 2 customers purchased 2 hamburgers. How many more hamburgers does Frank need to sell to make $50?\nAnswer: 4\n\nQuestion: There are 40 students in a class. If 1/10 are absent, 3/4 of the students who are present are in the classroom, and the rest are in the canteen, how many students are in the canteen?\nAnswer: 9\n

In [10]:
print(turbo.inspect_history(n=1))  # Print the latest response from the model




Given the fields `question`, produce the fields `answer`.

---

Follow the following format.

Question: ${question}
Reasoning: Let's think step by step in order to ${produce the answer}. We ...
Answer: ${answer}

---

Question: Andrew works in a company that provides a generous vacation allotment: for every 10 days worked, you get 1 vacation day. If last year Andrew worked 300 days and took 5 days off in March and twice as many in September, how many more vacation days can Andrew still take?
Answer: 15

Question: Frank is making hamburgers and he wants to sell them to make $50. Frank is selling each hamburger for $5 and 2 people purchased 4 and another 2 customers purchased 2 hamburgers. How many more hamburgers does Frank need to sell to make $50?
Answer: 4

Question: There are 40 students in a class. If 1/10 are absent, 3/4 of the students who are present are in the classroom, and the rest are in the canteen, how many students are in the canteen?
Answer: 9

Question: Betty and Dor

In [6]:
import dspy
from dspy.datasets.gsm8k import GSM8K, gsm8k_metric
# import mlflow

# # Enabling tracing for DSPy
# mlflow.dspy.autolog(log_compiles=True, log_evals=True, log_traces_from_compile=True)

# # Optional: Set a tracking URI and an experiment
# mlflow.set_tracking_uri("http://localhost:5000")
# mlflow.set_experiment("DSPy")

gsm8k = GSM8K()

trainset, devset = gsm8k.train[:50], gsm8k.dev[:50]

program = dspy.ChainOfThought("question -> answer")

# define a teleprompter
teleprompter = dspy.teleprompt.MIPROv2(
    metric=gsm8k_metric,
    auto="light",
)
# run the optimizer
optimized_program = teleprompter.compile(
    program,
    trainset=trainset,
    max_bootstrapped_demos=3,
    max_labeled_demos=4,
    requires_permission_to_run=False,
)

c:\Users\jayit\GenerativeAI\dspy\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 1319/1319 [00:00<00:00, 25579.28it/s]
2025/10/14 20:24:59 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '85e4264844594967a11b76cf3b10b316', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current dspy workflow
2025/10/14 20:24:59 WARNING dspy.teleprompt.mipro_optimizer_v2: 'requires_permission_to_run' is deprecated and will be removed in a future version.
2025/10/14 20:24:59 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 10
minibatch: False
num_fewshot_candidates: 6
num_instruct_candidates: 3
valset size: 40

2025/10/14 20:24:59 INFO dspy.teleprompt.mipro_optimizer_v2: 


Bootstrapping set 1/6
Bootstrapping set 2/6
Bootstrapping set 3/6


  0%|          | 0/10 [00:00<?, ?it/s]2025/10/14 20:24:59 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/14 20:25:14 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/14 20:25:14 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/14 20:25:14 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/14 20:25:14 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
 1

Bootstrapped 3 full traces after 3 examples for up to 1 rounds, amounting to 3 attempts.


Bootstrapping set 4/6


...

2025/10/14 23:38:38 INFO dspy.evaluate.evaluate: Average Metric: 35 / 40 (87.5%)


2025/10/14 23:38:38 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far! Score: 87.5
2025/10/14 23:38:38 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 87.5 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 5'].
2025/10/14 23:38:38 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [67.5, 82.5, 72.5, 82.5, 80.0, 85.0, 75.0, 77.5, 82.5, 77.5, 87.5]
2025/10/14 23:38:38 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 87.5
2025/10/14 23:38:38 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2025/10/14 23:38:38 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 87.5!


🏃 View run eval_full_10 at: http://localhost:5000/#/experiments/949896493763218950/runs/ed8746c0f0b549a4991d020abeb65fbb
🧪 View experiment at: http://localhost:5000/#/experiments/949896493763218950


🏃 View run stylish-rat-160 at: http://localhost:5000/#/experiments/949896493763218950/runs/85e4264844594967a11b76cf3b10b316
🧪 View experiment at: http://localhost:5000/#/experiments/949896493763218950


[Trace(trace_id=tr-ac74085bb5d9389372d3ffa3efab3ff1), Trace(trace_id=tr-91ca7b4bd099ee8fea1e04cd48d6630a), Trace(trace_id=tr-10ed01ebe722a64bbbdcc113ffee78d0), Trace(trace_id=tr-361180c1042b8151db06e12075d98a99), Trace(trace_id=tr-f63052ee4f112f2a2af6adb285a0fabb), Trace(trace_id=tr-e0ca14de9fde9dbbbaf69751ec169f58), Trace(trace_id=tr-219d4da3b6aeabd6655ecbc75fb52ec7), Trace(trace_id=tr-2eb6504a9e11afc05b80302654382d0f), Trace(trace_id=tr-04edb6da30eaf4330484e74fad57466a), Trace(trace_id=tr-c8e33b115e2574f75356fc0b4fcab9ef)]

In [7]:
# Save optimize program for future use
optimized_program.save(f"optimized.json")

In [ ]:
from dspy.evaluate import Evaluate

# 5. Evaluate the optimized program on the development set
# The evaluation reports the final score based on the gsm8k_metric.
print("Evaluating optimized program...")
evaluate_program = Evaluate(
    devset=devset,
    metric=gsm8k_metric,
    num_threads=4, # Use multiple threads for faster evaluation
    display_progress=True,
    display_table=10 # Display results for the first 10 examples
)

compiled_score = evaluate_program(optimized_program)

# 6. (Optional) Evaluate the unoptimized program for comparison
print("Evaluating unoptimized program...")
uncompiled_score = evaluate_program(CoT())

print(f"\nScore for optimized model: {compiled_score}")
print(f"Score for unoptimized model: {uncompiled_score}")

Evaluating optimized program...
  0%|          | 0/50 [00:00<?, ?it/s]

2025/10/15 04:25:00 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/15 04:25:00 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/15 04:25:00 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/15 04:25:00 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/15 04:25:34 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/15 04:25:34 WARNING mlflow.traci

Average Metric: 1.00 / 1 (100.0%):   2%|▏         | 1/50 [00:35<28:56, 35.44s/it]

2025/10/15 04:25:35 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/15 04:25:40 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/15 04:25:40 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/15 04:25:40 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/15 04:25:40 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}


Average Metric: 2.00 / 2 (100.0%):   4%|▍         | 2/50 [00:40<14:14, 17.80s/it]

...

2025/10/15 04:45:49 INFO dspy.evaluate.evaluate: Average Metric: 32 / 50 (64.0%)


,question,gold_reasoning,example_answer,reasoning,pred_answer,gsm8k_metric
0,"20 birds migrate on a seasonal basis from one lake to another, sea...",The birds' flying distance between Lake Jim through lake Disney to...,2200,"The birds first fly from lake Jim to lake Disney, a distance of 50...",The combined distance traveled by all the birds is 50 + 60 = 110 m...,
1,"Wendy went to the dentist for a cleaning, two fillings, and a toot...",Wendy’s dentist bill was 5 * $120 = $<<5*120=600>>600. She got two...,290,The dentist charges $70 for a cleaning and $120 for a filling. Wen...,$600 - $70 = $530,
2,Karen is packing her backpack for a long-distance hike. She packs ...,First find the weight of food Karen eats every hour: 2 pounds * 1/...,34,"To find out how much weight Karen is carrying after six hours, we ...",Karen is carrying 14 pounds of gear and the remaining food and wat...,
3,Rita is reading a five-chapter book with 95 pages. Each chapter ha...,Let P be the number of pages in the first chapter. The second chap...,13,"To find out how many pages the first chapter has, we need to work ...",The first chapter has 13 pages.,✔️ [True]
4,Benjamin collects 6 dozen eggs a day. Carla collects 3 times the n...,Carla:3(6)=18 dozen eggs Trisha: 6-4=<<6-4=2>>2 dozen eggs Total: ...,26,Carla collects 3 times the number of eggs that Benjamin collects. ...,26,✔️ [True]
5,Roy spends 2 hours on sports activities in school every day. He go...,He goes to.school 5 days a week so if he misses 2 days within the ...,6,To find out how many hours Roy spent on sports in school that week...,6 hours,✔️ [True]
6,Cameron is printing her thesis in the school library and has 400 A...,"If she had 400 papers, the total number of documents she separated...",240,Since 40% of the papers did not print out up to her desired qualit...,"To calculate 60% of 400, we multiply 400 by 0.6 (which is equivale...",
7,Burt spent $2.00 on a packet of basil seeds and $8.00 on potting s...,He spent $2.00 on seeds and $8.00 on soil for a total of 2+8 = $<<...,90,Burt's total cost is $2.00 + $8.00 = $10.00. He sells 20 basil pla...,$90.00,✔️ [True]
8,Martha's cat catches 3 rats and 7 birds. Cara's cat catches 3 less...,First find the total number of animals Martha's cat catches: 3 rat...,47,Cara's cat catches five times as many animals as Martha's cat. Fiv...,50,
9,Trey is raising money for a new bike that costs $112. He plans to ...,He needs to sell 112 bracelets because 112 divided by 1 equals <<1...,8,"To determine how many bracelets Trey needs to sell each day, we ne...",Trey needs to sell 8 bracelets each day.,✔️ [True]


🏃 View run eval at: http://localhost:5000/#/experiments/949896493763218950/runs/f8638c7572d942e1bc49686152914bd9
🧪 View experiment at: http://localhost:5000/#/experiments/949896493763218950

Score for optimized model: EvaluationResult(score=82.0, results=<list of 50 results>)
Score for unoptimized model: EvaluationResult(score=64.0, results=<list of 50 results>)


[Trace(trace_id=tr-6d1ed3419642e65f3fb47612421c1477), Trace(trace_id=tr-f0313d6f5d25fbb6f2d9c470c99bcea3), Trace(trace_id=tr-c65469da7c0b14a62cf7ab7c11818d68), Trace(trace_id=tr-7512824eb488a6aced45e8210d13478a), Trace(trace_id=tr-7b06fd43115b79ad29b7dee8f17795b2), Trace(trace_id=tr-daf37af57a508e3ce02cd13f3e75daf9), Trace(trace_id=tr-a2766eb848b042dbece59808e7edff4f), Trace(trace_id=tr-cfe0a9bc635e4e44edbe755f8a53616c), Trace(trace_id=tr-9dad6e7f6f2da96dc5ef4bc4cdabb0e6), Trace(trace_id=tr-719e6477aae8705790a8876a5d5a6e3e)]

: 

In [23]:
print(dspy.inspect_history())





[2025-10-14T14:31:41.363200]

System message:

Your input fields are:
1. `question` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## question ## ]]
{question}

Outputs will be a JSON object with the following fields.

{
  "reasoning": "{reasoning}",
  "answer": "{answer}"
}
In adhering to this structure, your objective is: 
        Given the fields `question`, produce the fields `answer`.


User message:

[[ ## question ## ]]
Dakota gets hit by a bus and has to spend 3 days in the hospital. The hospital charges her $900/day for her bed, $250/hour for two specialists to look at her 15 minutes each, and $1800 for the ambulance ride. How much is Dakota's medical bill?

Respond with a JSON object in the following order of fields: `reasoning`, then `answer`.


Response:

<think>
Okay, let's see. I need to calculate Dak

[Trace(trace_id=tr-26406577128199307aa8eef10ea32cd7), Trace(trace_id=tr-960f7495cb48695d05c70b4922a02e60), Trace(trace_id=tr-0f10a25425057b0ef789a1aa45f12c63), Trace(trace_id=tr-491b9909c800627b5a7f3215f0317cfb), Trace(trace_id=tr-15e1c23598904d1d221b19b66ec3fc3a), Trace(trace_id=tr-d7db09988946e9a9d25d5e78a3ad2fb9)]

In [25]:
print(dspy.inspect_history(n=10))





[2025-10-14T14:31:19.169815]

System message:

Your input fields are:
1. `question` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## question ## ]]
{question}

Outputs will be a JSON object with the following fields.

{
  "reasoning": "{reasoning}",
  "answer": "{answer}"
}
In adhering to this structure, your objective is: 
        Given the fields `question`, produce the fields `answer`.


User message:

[[ ## question ## ]]
A tank contains 6000 liters of water, 2000 liters evaporated, and then 3500 liters were drained by Bob. How many liters are in the tank if it now rains for 30 minutes and every 10 minutes 350 liters of rain are added to the tank?

Respond with a JSON object in the following order of fields: `reasoning`, then `answer`.


Response:

<think>
Okay, let's see. The problem is about a tank with wate